# Mean Shift Ablation Study for State Model

**Author:** Sid Potti  


## Objective

This notebook tests whether a simple **mean shift baseline** can match the performance of the full State transformer model. The goal is to understand:

1. **How much predictive power comes from learning the "average shift" per perturbation?**
2. **At what training epoch does the transformer surpass the mean shift baseline?**
3. **Can we initialize training with mean shifts to speed up convergence?**

## Hypothesis

The perturbation effect can be approximated as a **consistent shift in embedding space** for each (cell_type, perturbation) pair:

```
perturbed_embedding ≈ control_embedding + mean_shift
```

If this approximation is good, the mean shift baseline should achieve similar MMD scores to the State transformer.

## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
from tqdm import tqdm

# Import our mean shift implementation
from mean_shift_ablation import MeanShiftTable, create_pred_h5ad_for_mmd

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("Imports successful!")

## Data Configuration

Specify the paths to your training and test data, and the column names in your AnnData objects.

In [ ]:


# Data paths
TRAIN_DATA_PATH = "competition_support_set/train_data.h5ad"  # TODO: Update this path
TEST_DATA_PATH = "competition_support_set/test_data.h5ad"    # TODO: Update this path 

# Column names in adata.obs
CELL_TYPE_COL = "cell_type"        # Column containing cell type labels
PERT_COL = "target_gene"           # Column containing perturbation names
CONTROL_PERT = "non-targeting"     # Name of control/untreated perturbation

# Embedding location
EMBED_KEY = "X_hvg"                # Key in adata.obsm for embeddings, or None for adata.X

# Output paths
OUTPUT_DIR = Path("mean_shift_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# Output file for MMD evaluation
PRED_H5AD_PATH = OUTPUT_DIR / "pred_lms.h5ad"  # LMS = Latent Mean Shift

print(f"Configuration:")
print(f"  Train data: {TRAIN_DATA_PATH}")
print(f"  Test data (real.h5ad): {TEST_DATA_PATH}")
print(f"  Output pred file: {PRED_H5AD_PATH}")
print(f"  Cell type column: {CELL_TYPE_COL}")
print(f"  Perturbation column: {PERT_COL}")
print(f"  Control perturbation: {CONTROL_PERT}")
print(f"  Embedding key: {EMBED_KEY}")

## Step 1: Load and Inspect Training Data

First, let's load the training data and verify its structure.

In [ ]:
print("Loading training data...")
adata_train = sc.read_h5ad(TRAIN_DATA_PATH)

print(f"\nTraining data shape: {adata_train.shape}")
print(f"  - Cells: {adata_train.n_obs:,}")
print(f"  - Features: {adata_train.n_vars:,}")

print(f"\nColumns in adata.obs: {list(adata_train.obs.columns)}")
print(f"Keys in adata.obsm: {list(adata_train.obsm.keys())}")

# Get embeddings
if EMBED_KEY and EMBED_KEY in adata_train.obsm:
    train_embeddings = adata_train.obsm[EMBED_KEY]
    print(f"\nUsing embeddings from adata.obsm['{EMBED_KEY}']")
else:
    train_embeddings = adata_train.X.toarray() if hasattr(adata_train.X, 'toarray') else adata_train.X
    print(f"\nUsing embeddings from adata.X")

print(f"Embedding shape: {train_embeddings.shape}")
embedding_dim = train_embeddings.shape[1]

# Verify required columns exist
assert CELL_TYPE_COL in adata_train.obs.columns, f"Column '{CELL_TYPE_COL}' not found in adata.obs"
assert PERT_COL in adata_train.obs.columns, f"Column '{PERT_COL}' not found in adata.obs"

# Get unique values
unique_cell_types = adata_train.obs[CELL_TYPE_COL].unique()
unique_perts = adata_train.obs[PERT_COL].unique()

print(f"\nUnique cell types: {len(unique_cell_types)}")
print(f"Cell types: {list(unique_cell_types)}")

print(f"\nUnique perturbations: {len(unique_perts)}")
print(f"First 10 perturbations: {list(unique_perts[:10])}")

# Check for control perturbation
assert CONTROL_PERT in unique_perts, f"Control perturbation '{CONTROL_PERT}' not found in data"
n_control = (adata_train.obs[PERT_COL] == CONTROL_PERT).sum()
print(f"\nControl cells ('{CONTROL_PERT}'): {n_control:,} ({n_control/adata_train.n_obs*100:.1f}%)")

## Step 2: Compute Mean Shift Table

For each (cell_type, perturbation) pair, we compute:

```python
control_mean = mean(control_embeddings for this cell_type)
perturbed_mean = mean(perturbed_embeddings for this cell_type + perturbation)
shift = perturbed_mean - control_mean
```

This shift vector captures the "average effect" of applying that perturbation to that cell type.

In [ ]:
print("Computing mean shift table...")

shift_table = MeanShiftTable()
shift_table.compute_from_anndata(
    adata_path=TRAIN_DATA_PATH,
    control_pert=CONTROL_PERT,
    cell_type_col=CELL_TYPE_COL,
    pert_col=PERT_COL,
    embed_key=EMBED_KEY
)

# Save the shift table
shift_table_path = OUTPUT_DIR / "mean_shift_table.pkl"
shift_table.save(str(shift_table_path))

print(f"\nMean shift table saved to: {shift_table_path}")

## Step 3: Verify Shift Table Structure

Let's verify that the computed shifts have the correct dimensions and structure.

In [ ]:
print("="*60)
print("SHIFT TABLE VERIFICATION")
print("="*60)

# Check number of shifts computed
n_shifts = len(shift_table.shifts)
n_control_means = len(shift_table.control_means)

print(f"\nNumber of shifts computed: {n_shifts}")
print(f"Number of cell types with control means: {n_control_means}")

# Verify all shifts have correct dimensions
print(f"\nVerifying shift dimensions...")
for key, shift in shift_table.shifts.items():
    assert shift.shape == (embedding_dim,), f"Shift {key} has wrong shape: {shift.shape}"
print(f"✓ All {n_shifts} shifts have correct shape: ({embedding_dim},)")

# Verify control means have correct dimensions
print(f"\nVerifying control mean dimensions...")
for cell_type, mean in shift_table.control_means.items():
    assert mean.shape == (embedding_dim,), f"Control mean for {cell_type} has wrong shape: {mean.shape}"
print(f"✓ All {n_control_means} control means have correct shape: ({embedding_dim},)")

# Show example shifts
print(f"\nExample shifts:")
for i, (key, shift) in enumerate(list(shift_table.shifts.items())[:3]):
    cell_type, pert = key
    magnitude = np.linalg.norm(shift)
    n_control, n_pert = shift_table.n_samples[key]
    print(f"\n  {i+1}. Cell type: {cell_type}")
    print(f"     Perturbation: {pert}")
    print(f"     Shift magnitude: {magnitude:.4f}")
    print(f"     Sample sizes: {n_control} control, {n_pert} perturbed")
    print(f"     First 5 shift values: {shift[:5]}")

print("\n✓ All assertions passed!")

## Step 4: Analyze Shift Statistics

Let's examine the distribution of shift magnitudes across different perturbations.

In [ ]:
# Get statistics
stats_df = shift_table.get_statistics()

print("Shift magnitude statistics:")
print(stats_df['shift_magnitude'].describe())

print("\nTop 20 perturbations by shift magnitude:")
print(stats_df.head(20))

# Save statistics
stats_path = OUTPUT_DIR / "shift_statistics.csv"
stats_df.to_csv(stats_path, index=False)
print(f"\nStatistics saved to: {stats_path}")

In [ ]:
# Plot distribution of shift magnitudes
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(stats_df['shift_magnitude'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(stats_df['shift_magnitude'].median(), color='r', linestyle='--', 
                label=f'Median: {stats_df["shift_magnitude"].median():.4f}')
axes[0].set_xlabel('Shift Magnitude (L2 Norm)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Distribution of Shift Magnitudes', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Boxplot by cell type
stats_df.boxplot(column='shift_magnitude', by='cell_type', ax=axes[1])
axes[1].set_xlabel('Cell Type', fontsize=12)
axes[1].set_ylabel('Shift Magnitude', fontsize=12)
axes[1].set_title('Shift Magnitudes by Cell Type', fontsize=14)
plt.suptitle('')  # Remove default title

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "shift_magnitude_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Plot saved to: {OUTPUT_DIR / 'shift_magnitude_distribution.png'}")

## Step 5: Create Prediction H5AD File for MMD Evaluation

Now we apply the mean shift to each test cell and save the predictions in a format compatible with `mmd_anndata_pair.py`.

**For each cell in the test set:**
- If it's a control cell: Keep original embedding
- If it's a perturbed cell: Apply mean shift based on (cell_type, perturbation)

The output file will have:
- Same cells and metadata as the input test file
- Predictions stored in `.obsm['model_preds']`

In [ ]:
create_pred_h5ad_for_mmd(
    adata_test_path=TEST_DATA_PATH,
    shift_table=shift_table,
    output_path=str(PRED_H5AD_PATH),
    control_pert=CONTROL_PERT,
    cell_type_col=CELL_TYPE_COL,
    pert_col=PERT_COL,
    embed_key=EMBED_KEY,
    pred_embed_key="model_preds"  # This is what mmd_anndata_pair.py expects
)

In [ ]:
import subprocess

# MMD evaluation configuration
MMD_OUTPUT_DIR = OUTPUT_DIR / "mmd_evaluation"
MMD_OUTPUT_DIR.mkdir(exist_ok=True)

# Build the command
mmd_command = [
    "python", "../scripts/mmd_state_pipeline/mmd_anndata_pair.py",
    "--adata-real", TEST_DATA_PATH,
    "--adata-pred", str(PRED_H5AD_PATH),
    "--control-pert", CONTROL_PERT,
    "--pert-col", PERT_COL,
    "--celltype-col", CELL_TYPE_COL,
    "--embed-key", EMBED_KEY,
    "--embed-key-pred", "model_preds",
    "--outdir", str(MMD_OUTPUT_DIR)
]

print("Running MMD evaluation...")
print(f"Command: {' '.join(mmd_command)}\n")

# Run the command
result = subprocess.run(mmd_command, capture_output=True, text=True)

# Print output
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

if result.returncode == 0:
    print(f"\n✓ MMD evaluation complete!")
    print(f"Results saved to: {MMD_OUTPUT_DIR}")
else:
    print(f"\n✗ MMD evaluation failed with return code {result.returncode}")



### Whats Done

1. ✓ Loaded training data and computed mean shift table for all (cell_type, perturbation) pairs
2. ✓ Analyzed shift statistics and visualized shift magnitude distributions
3. ✓ Created `pred_lms.h5ad` with mean shift predictions for test data
4. ✓ Output file is ready for MMD evaluation

### Next Steps

1. **Run MMD evaluation** using the command above
2. **Compare with State model MMD scores**:
   - If Transport MMD is similar → Mean shift captures most of the effect
   - If Transport MMD is much worse → Transformer learns additional complexity
4. **Test at different training epochs**: When does State surpass mean shift baseline?
